# Розширення генерації на основі пошуку (RAG) та векторні бази даних

In [21]:
%pip install getenv openai faiss-cpu pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [22]:
import os
import pandas as pd
import numpy as np
import faiss

from dotenv import load_dotenv

load_dotenv()

True

## Створення нашої бази знань

Налаштування FAISS для векторного пошуку

In [23]:

# Шляхи до файлів (даних для RAG)
data_paths = [
    "data/frameworks.md", 
    "data/own_framework.md", 
    "data/perceptron.md"
]
# Ініціалізація порожнього DataFrame
df = pd.DataFrame(columns=['path', 'text'])
# Сучасний спосіб додавання рядків до DataFrame
for path in data_paths:
    try:
        with open(path, 'r', encoding='utf-8') as file:
            file_content = file.read()
        # Використовуємо concat замість застарілого append
        new_row = pd.DataFrame({'path': [path], 'text': [file_content]})
        df = pd.concat([df, new_row], ignore_index=True)
    except FileNotFoundError:
        print(f"Файл не знайдено: {path}")
df.head()

,path,text
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...
1,data/own_framework.md,# Introduction to Neural Networks. Multi-Layer...
2,data/perceptron.md,# Introduction to Neural Networks: Perceptron\...


In [24]:
def split_text(text, max_length, min_length):
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if len(' '.join(current_chunk)) < max_length and len(' '.join(current_chunk)) > min_length:
            chunks.append(' '.join(current_chunk))
            current_chunk = []

    # Якщо останній фрагмент не досягнув мінімальної довжини, все одно додати його
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

# Припускаючи, що analyzed_df - це pandas DataFrame, а 'output_content' - це стовпець у цьому DataFrame
splitted_df = df.copy()
splitted_df['chunks'] = splitted_df['text'].apply(lambda x: split_text(x, 400, 300))

splitted_df

,path,text,chunks
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,[# Neural Network Frameworks As we have learne...
1,data/own_framework.md,# Introduction to Neural Networks. Multi-Layer...,[# Introduction to Neural Networks. Multi-Laye...
2,data/perceptron.md,# Introduction to Neural Networks: Perceptron\...,[# Introduction to Neural Networks: Perceptron...


In [25]:
# Припускаючи, що 'chunks' - це стовпець списків у DataFrame splitted_df, ми розділимо фрагменти на різні рядки
flattened_df = splitted_df.explode('chunks')

flattened_df.head()


,path,text,chunks
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,# Neural Network Frameworks As we have learned...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,descent optimization While the `numpy` library...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,should give us the opportunity to compute grad...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,those computations on GPUs is very important. ...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"API, there is also higher-level API, called Ke..."


## Перетворення тексту на ембедінги

In [26]:
from azure.ai.inference import EmbeddingsClient
from azure.core.credentials import AzureKeyCredential

endpoint = "https://models.inference.ai.azure.com"
token = os.getenv("GITHUB_TOKEN")

embed_model_name = "cohere-embed-v3-multilingual" 

embed_client = EmbeddingsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(token)
)

def create_embeddings(text):
    """
    Створює ембедінги для тексту.
    
    Args:
        text: Текст або список текстів для ембедінгу
        model: Модель для ембедінгу (за замовчуванням mistral-embed)
        
    Returns:
        Вектор ембедінгу
    """
    # Обробка pandas Series
    if isinstance(text, pd.Series):
        # Беремо перший елемент з Series
        text = text.iloc[0]
    
    # Перетворюємо в список рядків для API
    if not isinstance(text, list):
        text = [str(text)]
    else:
        text = [str(item) for item in text]
    
    embeddings_response = embed_client.embed(
        input=text,
        model=embed_model_name
    )
    
    # Повернення ембедінгу для першого елемента
    return embeddings_response.data[0].embedding

# Приклад використання:
embeddings = create_embeddings(flattened_df['chunks'][0])

In [27]:
cat = create_embeddings("cat")

cat

[0.00630188,
 0.010032654,
 -0.0020694733,
 -0.015914917,
 -0.02796936,
 0.007827759,
 -0.005268097,
 -0.036468506,
 -0.005508423,
 0.0060577393,
 0.023132324,
 0.013717651,
 0.0005393028,
 0.010406494,
 0.005924225,
 0.009422302,
 0.03390503,
 -0.021362305,
 0.010467529,
 -0.011383057,
 -0.0012874603,
 0.018737793,
 0.042938232,
 0.0024642944,
 -0.036132812,
 0.043304443,
 0.016082764,
 -0.036895752,
 0.018218994,
 -0.029144287,
 -0.016052246,
 -0.0056266785,
 0.027389526,
 0.028289795,
 -0.022781372,
 0.021118164,
 -0.022994995,
 -0.049987793,
 0.0027751923,
 0.04925537,
 0.010757446,
 0.02684021,
 -0.020858765,
 0.023529053,
 -0.06488037,
 0.014717102,
 0.013954163,
 0.028549194,
 0.026428223,
 0.023391724,
 0.00060892105,
 0.002544403,
 0.023666382,
 -0.002866745,
 0.027145386,
 -0.00818634,
 0.002204895,
 0.0050354004,
 -0.0060272217,
 -0.0025501251,
 0.03152466,
 0.016815186,
 -0.034301758,
 0.04663086,
 -0.014587402,
 0.06756592,
 0.043914795,
 0.03741455,
 0.018051147,
 0.01948

In [28]:
import pickle
from pathlib import Path

def save_embeddings(df, folder="embeddings", filename="flattened_df.pkl"):
    """
    Зберігає DataFrame з ембедінгами в указану папку.
    
    Args:
        df: DataFrame з ембедінгами
        folder: Назва папки для збереження
        filename: Ім'я файлу для збереження
    """
    # Створення директорії, якщо вона не існує
    Path(folder).mkdir(parents=True, exist_ok=True)
    
    # Шлях до файлу
    file_path = os.path.join(folder, filename)
    
    # Збереження DataFrame
    with open(file_path, 'wb') as f:
        pickle.dump(df, f)
    
    print(f"DataFrame успішно збережено в {file_path}")

def load_embeddings(folder="embeddings", filename="flattened_df.pkl"):
    """
    Завантажує DataFrame з ембедінгами з указаної папки.
    
    Args:
        folder: Назва папки для завантаження
        filename: Ім'я файлу для завантаження
        
    Returns:
        DataFrame з ембедінгами або None, якщо файл не існує
    """
    # Шлях до файлу
    file_path = os.path.join(folder, filename)
    
    # Перевірка існування файлу
    if os.path.exists(file_path):
        # Завантаження DataFrame
        with open(file_path, 'rb') as f:
            df = pickle.load(f)
        
        print(f"DataFrame успішно завантажено з {file_path}")
        return df
    else:
        print(f"Файл {file_path} не знайдено")
        return None

def get_or_create_embeddings(df, chunk_column, embedding_function, folder="embeddings", filename="flattened_df.pkl"):
    """
    Завантажує DataFrame з ембедінгами або створює новий.
    
    Args:
        df: Вихідний DataFrame з текстами
        chunk_column: Назва стовпця з текстовими фрагментами
        embedding_function: Функція для створення ембедінгів
        folder: Назва папки для збереження/завантаження
        filename: Ім'я файлу для збереження/завантаження
        
    Returns:
        DataFrame з ембедінгами
    """
    # Спроба завантажити DataFrame
    loaded_df = load_embeddings(folder, filename)
    
    if loaded_df is not None:
        return loaded_df
    
    # Якщо завантаження не вдалося, створюємо ембедінги
    print("Створення нових ембедінгів...")
    
    # Створення ембедінгів
    embeddings = []
    for chunk in df[chunk_column]:
        embeddings.append(embedding_function(chunk))
    
    # Збереження ембедінгів в DataFrame
    df['embeddings'] = embeddings
    
    # Збереження DataFrame
    save_embeddings(df, folder, filename)
    
    return df


# Використовуємо функцію, яка розраховує ембедінги, 
# якщо вони не були раніше створені і збережені в папці в "15-rag-and-vector-databases/embeddings"
flattened_df = get_or_create_embeddings(
    splitted_df.explode('chunks'), 
    'chunks', 
    create_embeddings
)

flattened_df.head()

DataFrame успішно завантажено з embeddings\flattened_df.pkl


,path,text,chunks,embeddings
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,# Neural Network Frameworks As we have learned...,"[0.005393982, 0.018859863, 0.0063819885, 0.009..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,descent optimization While the `numpy` library...,"[0.01537323, 0.0284729, -0.02949524, 0.0531921..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,should give us the opportunity to compute grad...,"[0.016677856, -0.004032135, -0.01838684, 0.051..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,those computations on GPUs is very important. ...,"[-0.0016927719, -0.01461792, 0.013427734, 0.03..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"API, there is also higher-level API, called Ke...","[-0.018341064, -0.008888245, -0.017364502, 0.0..."


# Пошук з використанням FAISS

Векторний пошук та схожість між нашим запитом і базою даних з використанням FAISS

### Створення індексу FAISS та підготовка до пошуку

In [29]:
# Отримуємо ембедінги як масив numpy
embeddings_list = flattened_df['embeddings'].to_list()
embeddings_array = np.array(embeddings_list).astype('float32')

# Визначаємо розмірність векторів
vector_dimension = len(embeddings_list[0])

# Створюємо індекс FAISS
index = faiss.IndexFlatL2(vector_dimension)  # L2 - це евклідова відстань

# Додаємо наші вектори до індексу
index.add(embeddings_array)

# Перевіряємо кількість векторів в індексі
print(f"Загальна кількість векторів в індексі: {index.ntotal}, розмірність векторів: {vector_dimension}")

Загальна кількість векторів в індексі: 57, розмірність векторів: 1024


In [30]:
def search_test(question, k):
    # question - Ваше текстове запитання
    # Перетворіть запитання у вектор запиту
    query_vector = create_embeddings(question)

    query_vector_array = np.array([query_vector]).astype('float32')

    # Знайдіть найбільш схожі документи (k=5 - скільки найближчих сусідів шукаємо)
    distances, indices = index.search(query_vector_array, k)

    # Виведіть найбільш схожі документи
    for i in range(min(3, len(indices[0]))):
        idx = indices[0][i]
        print(f"Фрагмент {i+1}:")
        print(flattened_df['chunks'].iloc[idx])
        print(f"Шлях: {flattened_df['path'].iloc[idx]}")
        print(f"Відстань: {distances[0][i]}")
        print("-" * 50)
search_test("Що таке перциптрон", 5)

Фрагмент 1:
user to adjust the resistance of a circuit. > The New York Times wrote about perceptron at that time: *the embryo of an electronic computer that [the Navy] expects will be able to walk, talk, see, write, reproduce itself and be conscious of its existence.* ## Perceptron Model Suppose we have N features
Шлях: data/perceptron.md
Відстань: 0.9729086756706238
--------------------------------------------------
Фрагмент 2:
# Introduction to Neural Networks. Multi-Layered Perceptron In the previous section, you learned about the simplest neural network model - one-layered perceptron, a linear two-class classification model. In this section we will extend this model into a more flexible framework, allowing us to: * perform
Шлях: data/own_framework.md
Відстань: 1.0899983644485474
--------------------------------------------------
Фрагмент 3:
in our model, in which case the input vector would be a vector of size N. A perceptron is a **binary classification** model, i.e. it can distin

## Поєднання всього для відповіді на запитання

In [31]:
from azure.ai.inference import ChatCompletionsClient

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

# Виберіть модель загального призначення для тексту
deployment = "gpt-4o-mini"

# Реалізація чатботів (при наявності і відсутності RAG)

In [32]:
def chatbot_with_rag(user_input, k = 5):
    # Перетворіть запитання у вектор запиту
    query_vector = create_embeddings(user_input)
    query_vector_array = np.array([query_vector]).astype('float32')
    
    # Знайдіть найбільш схожі документи з FAISS
    # k - кількість найближчих сусідів для пошуку
    distances, indices = index.search(query_vector_array, k)

    # додайте документи до запиту, щоб забезпечити контекст
    history = []
    for idx in indices[0]:
        history.append(flattened_df['chunks'].iloc[idx])

    # створюємо об'єкт повідомлення з контекстом
    context = "\n\n".join(history)  # всі знайдені фрагменти

    # Формуємо запит, що просить коротку, але завершену відповідь
    messages = [
        {"role": "system", "content": "You are an AI assistant that helps with AI questions. "},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_input}\n\n Provide a brief but complete answer based on the context. Answer in Ukrainian."}
    ]


    response = client.complete(
        temperature=0,
        model=deployment,
        messages=messages,
        max_tokens=300,
    )

    return response.choices[0].message.content


def chatbot_without_rag(user_input):
    """
    Чат-бот без використання RAG (прямий запит до моделі).
    """
    messages=[
        {"role": "system", "content": "You are an AI assistant that helps with AI questions. Provide brief but complete answers. Answer in Ukrainian."},
        {"role": "user", "content": f"Question: {user_input}"}
    ]

    response = client.complete(
        temperature=0,
        model=deployment,
        messages=messages,
        max_tokens=300,
    )

    return response.choices[0].message.content

# Порівняння відповідей RAG-системи

In [33]:
from IPython.display import display, Markdown, HTML

def compare_responses(user_input, save_to_file=False, filename="rag_comparison.md", k = 5):
    """
    Порівнює відповіді чат-боту з RAG та без RAG.
    
    Args:
        user_input: Запитання користувача
        save_to_file: Зберегти результат у файл Markdown
        filename: Назва файлу для збереження
    """
    # Отримання знайдених чанків
    query_vector = create_embeddings(user_input)

    query_vector_array = np.array([query_vector]).astype('float32')
    # k - кількість найближчих сусідів для пошуку
    distances, indices = index.search(query_vector_array, k)
    
    # Отримання відповідей
    rag_response = chatbot_with_rag(user_input)
    no_rag_response = chatbot_without_rag(user_input)
    
    # Формування markdown-тексту
    markdown_text = f"""
# Порівняння відповідей

## 📝 Запит: {user_input}

## 🔍 Відповіді моделей

### Без використання RAG

{no_rag_response}

### З використанням RAG

{rag_response}

## 📚 Знайдені фрагменти тексту

"""
    
    # Додавання чанків
    for i, idx in enumerate(indices[0]):
        chunk_content = flattened_df['chunks'].iloc[idx]
        path = flattened_df['path'].iloc[idx]
        dist = float(distances[0][i])
        
        markdown_text += f"""
### Фрагмент {i+1} (відстань: {dist:.4f})

**Шлях**: {path}

{chunk_content}

"""
    
    # Виведення Markdown
    display(Markdown(markdown_text))
    
    # Для коректного відображення формул
    mathjax_script = """
    <script type="text/javascript">
        MathJax = {
            tex: {
                inlineMath: [['$', '$']]
            }
        };
    </script>
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js">
    </script>
    """
    display(HTML(mathjax_script))
    
    # Збереження в файл, якщо потрібно
    if save_to_file:
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(markdown_text)
        print(f"Результати збережено у файл: {filename}")


In [34]:
# Приклад використання:
compare_responses("Що таке перцептрон?")


# Порівняння відповідей

## 📝 Запит: Що таке перцептрон?

## 🔍 Відповіді моделей

### Без використання RAG

Перцептрон — це базова модель штучного нейрону, яка використовується в машинному навчанні для класифікації даних. Він складається з вхідних сигналів, ваг, активаційної функції та виходу. Перцептрон навчається шляхом корекції ваг на основі помилок, що виникають під час прогнозування. Це один з найпростіших типів нейронних мереж і слугує основою для більш складних архітектур.

### З використанням RAG

Перцептрон — це модель для бінарної класифікації, яка може розрізняти два класи вхідних даних. Він представляє собою просту нейронну мережу, що складається з одного шару, і був розроблений Френком Розенблаттом у 1957 році. Перцептрон приймає вектор вхідних даних і видає результат у вигляді +1 або -1, в залежності від класу. Ця модель стала основою для подальшого розвитку більш складних нейронних мереж.

## 📚 Знайдені фрагменти тексту


### Фрагмент 1 (відстань: 0.9221)

**Шлях**: data/perceptron.md

user to adjust the resistance of a circuit. > The New York Times wrote about perceptron at that time: *the embryo of an electronic computer that [the Navy] expects will be able to walk, talk, see, write, reproduce itself and be conscious of its existence.* ## Perceptron Model Suppose we have N features


### Фрагмент 2 (відстань: 0.9672)

**Шлях**: data/perceptron.md

in our model, in which case the input vector would be a vector of size N. A perceptron is a **binary classification** model, i.e. it can distinguish between two classes of input data. We will assume that for each input vector x the output of our perceptron would be either +1 or -1, depending on the class.


### Фрагмент 3 (відстань: 0.9851)

**Шлях**: data/own_framework.md

# Introduction to Neural Networks. Multi-Layered Perceptron In the previous section, you learned about the simplest neural network model - one-layered perceptron, a linear two-class classification model. In this section we will extend this model into a more flexible framework, allowing us to: * perform


### Фрагмент 4 (відстань: 1.0372)

**Шлях**: data/perceptron.md

# Introduction to Neural Networks: Perceptron One of the first attempts to implement something similar to a modern neural network was done by Frank Rosenblatt from Cornell Aeronautical Laboratory in 1957. It was a hardware implementation called "Mark-1", designed to recognize primitive geometric figures,


### Фрагмент 5 (відстань: 1.0552)

**Шлях**: data/perceptron.md

and to continue learning - go to Perceptron notebook. Here's an interesting article about perceptrons as well. ## Assignment In this lesson, we have implemented a perceptron for binary classification task, and we have used it to classify between two handwritten digits. In this lab, you are asked to solve



___

## Практичні завдання
1. Запуск базового прикладу - виконайте код з ноутбука та проаналізуйте результати.
2. Тестування з різними запитами - спробуйте різні запитання до системи та порівняйте відповіді з RAG та без нього.
3. Модифікація параметрів чанкінгу - змініть параметри функції split_text (розмір фрагментів) та проаналізуйте, як це впливає на якість відповідей.
4. Зміна кількості фрагментів - змініть параметр k у функції пошуку та дослідіть, як кількість знайдених фрагментів впливає на відповіді.

### 1. Запуск базового прикладу - виконайте код з ноутбука та проаналізуйте результати.

### 2. Тестування з різними запитами - спробуйте різні запитання до системи та порівняйте відповіді з RAG та без нього.

In [35]:
# Виконуємо порівняння для нового запиту
compare_responses("Що таке згорткові нейронні мережі?", save_to_file=True, filename="neural_nework.md")


# Порівняння відповідей

## 📝 Запит: Що таке згорткові нейронні мережі?

## 🔍 Відповіді моделей

### Без використання RAG

Згорткові нейронні мережі (CNN, Convolutional Neural Networks) — це тип глибоких нейронних мереж, які спеціально розроблені для обробки даних з сітковою структурою, таких як зображення. Вони використовують згорткові шари для автоматичного виділення ознак, що дозволяє ефективно розпізнавати об'єкти, текстури та інші патерни в зображеннях. CNN зазвичай складаються з кількох згорткових шарів, шарів підвибірки (пулінгу) та повнозв'язних шарів, що робить їх потужними для задач комп'ютерного зору.

### З використанням RAG

Згорткові нейронні мережі (CNN) — це тип нейронних мереж, які спеціально розроблені для обробки даних з решітковою структурою, таких як зображення. Вони використовують згорткові шари для автоматичного виділення ознак з вхідних даних, що дозволяє ефективно виконувати завдання, пов'язані з класифікацією, розпізнаванням об'єктів та іншими задачами комп'ютерного зору. Згорткові мережі здатні виявляти складні патерни та структури в даних, що робить їх потужним інструментом у сфері штучного інтелекту.

## 📚 Знайдені фрагменти тексту


### Фрагмент 1 (відстань: 0.9587)

**Шлях**: data/own_framework.md

# Introduction to Neural Networks. Multi-Layered Perceptron In the previous section, you learned about the simplest neural network model - one-layered perceptron, a linear two-class classification model. In this section we will extend this model into a more flexible framework, allowing us to: * perform


### Фрагмент 2 (відстань: 0.9745)

**Шлях**: data/perceptron.md

array, so the neural network had 400 inputs and one binary output. A simple network contained one neuron, also called a **threshold logic unit**. Neural network weights acted like potentiometers that required manual adjustment during the training phase. > ✅ A potentiometer is a device that allows the


### Фрагмент 3 (відстань: 0.9998)

**Шлях**: data/own_framework.md

**multi-class classification** in addition to two-class * solve **regression problems** in addition to classification * separate classes that are not linearly separable We will also develop our own modular framework in Python that will allow us to construct different neural network architectures. ## Formalization


### Фрагмент 4 (відстань: 1.0301)

**Шлях**: data/own_framework.md

data. Because subset is taken randomly each time, such method is called **stochastic gradient descent** (SGD). ## Multi-Layered Perceptrons and Backpropagation One-layer network, as we have seen above, is capable of classifying linearly separable classes. To build a richer model, we can combine several


### Фрагмент 5 (відстань: 1.0328)

**Шлях**: data/perceptron.md

# Introduction to Neural Networks: Perceptron One of the first attempts to implement something similar to a modern neural network was done by Frank Rosenblatt from Cornell Aeronautical Laboratory in 1957. It was a hardware implementation called "Mark-1", designed to recognize primitive geometric figures,



Результати збережено у файл: neural_nework.md


### 3. Модифікація параметрів чанкінгу - змініть параметри функції split_text (розмір фрагментів) та проаналізуйте, як це впливає на якість відповідей.

In [36]:
splitted_df['chunks'] = splitted_df['text'].apply(lambda x: split_text(x, 500, 350))
compare_responses("Що таке згорткові нейронні мережі?", save_to_file=True, filename="split_text_500_350.md")


# Порівняння відповідей

## 📝 Запит: Що таке згорткові нейронні мережі?

## 🔍 Відповіді моделей

### Без використання RAG

Згорткові нейронні мережі (CNN, Convolutional Neural Networks) — це тип глибоких нейронних мереж, які спеціально розроблені для обробки даних з сітковою структурою, таких як зображення. Вони використовують згорткові шари для автоматичного виділення ознак, що дозволяє ефективно розпізнавати об'єкти, текстури та інші патерни в зображеннях. CNN зазвичай складаються з кількох згорткових шарів, шарів підвибірки (пулінгу) та повнозв'язних шарів, що робить їх потужними для задач комп'ютерного зору.

### З використанням RAG

Згорткові нейронні мережі (CNN) — це тип нейронних мереж, які спеціально розроблені для обробки даних з решітковою структурою, таких як зображення. Вони використовують згорткові шари для автоматичного виділення ознак з вхідних даних, що дозволяє ефективно виконувати завдання, пов'язані з класифікацією зображень, виявленням об'єктів та іншими задачами комп'ютерного зору. Згорткові мережі зазвичай складаються з кількох згорткових шарів, які слідують за шарами підвибірки, що зменшує розмірність даних і зберігає важливу інформацію.

## 📚 Знайдені фрагменти тексту


### Фрагмент 1 (відстань: 0.9587)

**Шлях**: data/own_framework.md

# Introduction to Neural Networks. Multi-Layered Perceptron In the previous section, you learned about the simplest neural network model - one-layered perceptron, a linear two-class classification model. In this section we will extend this model into a more flexible framework, allowing us to: * perform


### Фрагмент 2 (відстань: 0.9745)

**Шлях**: data/perceptron.md

array, so the neural network had 400 inputs and one binary output. A simple network contained one neuron, also called a **threshold logic unit**. Neural network weights acted like potentiometers that required manual adjustment during the training phase. > ✅ A potentiometer is a device that allows the


### Фрагмент 3 (відстань: 0.9998)

**Шлях**: data/own_framework.md

**multi-class classification** in addition to two-class * solve **regression problems** in addition to classification * separate classes that are not linearly separable We will also develop our own modular framework in Python that will allow us to construct different neural network architectures. ## Formalization


### Фрагмент 4 (відстань: 1.0301)

**Шлях**: data/own_framework.md

data. Because subset is taken randomly each time, such method is called **stochastic gradient descent** (SGD). ## Multi-Layered Perceptrons and Backpropagation One-layer network, as we have seen above, is capable of classifying linearly separable classes. To build a richer model, we can combine several


### Фрагмент 5 (відстань: 1.0328)

**Шлях**: data/perceptron.md

# Introduction to Neural Networks: Perceptron One of the first attempts to implement something similar to a modern neural network was done by Frank Rosenblatt from Cornell Aeronautical Laboratory in 1957. It was a hardware implementation called "Mark-1", designed to recognize primitive geometric figures,



Результати збережено у файл: split_text_500_350.md


In [37]:
# Для коротких документів
splitted_df['chunks'] = splitted_df['text'].apply(lambda x: split_text(x, 300, 200))
compare_responses("Що таке згорткові нейронні мережі?", save_to_file=True, filename="split_text_300_200.md")


# Порівняння відповідей

## 📝 Запит: Що таке згорткові нейронні мережі?

## 🔍 Відповіді моделей

### Без використання RAG

Згорткові нейронні мережі (CNN, Convolutional Neural Networks) — це тип глибоких нейронних мереж, які спеціально розроблені для обробки даних з сітковою структурою, таких як зображення. Вони використовують згорткові шари для автоматичного виділення ознак, що дозволяє ефективно розпізнавати об'єкти, текстури та інші елементи в зображеннях. CNN зазвичай складаються з кількох згорткових шарів, шарів підвибірки (пулінгу) та повнозв'язних шарів, що робить їх потужними для задач комп'ютерного зору, таких як класифікація зображень, детекція об'єктів та сегментація.

### З використанням RAG

Згорткові нейронні мережі (CNN) — це тип нейронних мереж, які спеціально розроблені для обробки даних з решітчастою структурою, таких як зображення. Вони використовують згорткові шари для автоматичного виділення ознак з вхідних даних, що дозволяє ефективно виконувати завдання, пов'язані з класифікацією, розпізнаванням об'єктів та іншими задачами комп'ютерного зору. Згорткові мережі зазвичай складаються з кількох згорткових шарів, які слідують за шарами підвибірки, що зменшує розмірність даних і зберігає важливу інформацію.

## 📚 Знайдені фрагменти тексту


### Фрагмент 1 (відстань: 0.9587)

**Шлях**: data/own_framework.md

# Introduction to Neural Networks. Multi-Layered Perceptron In the previous section, you learned about the simplest neural network model - one-layered perceptron, a linear two-class classification model. In this section we will extend this model into a more flexible framework, allowing us to: * perform


### Фрагмент 2 (відстань: 0.9745)

**Шлях**: data/perceptron.md

array, so the neural network had 400 inputs and one binary output. A simple network contained one neuron, also called a **threshold logic unit**. Neural network weights acted like potentiometers that required manual adjustment during the training phase. > ✅ A potentiometer is a device that allows the


### Фрагмент 3 (відстань: 0.9998)

**Шлях**: data/own_framework.md

**multi-class classification** in addition to two-class * solve **regression problems** in addition to classification * separate classes that are not linearly separable We will also develop our own modular framework in Python that will allow us to construct different neural network architectures. ## Formalization


### Фрагмент 4 (відстань: 1.0301)

**Шлях**: data/own_framework.md

data. Because subset is taken randomly each time, such method is called **stochastic gradient descent** (SGD). ## Multi-Layered Perceptrons and Backpropagation One-layer network, as we have seen above, is capable of classifying linearly separable classes. To build a richer model, we can combine several


### Фрагмент 5 (відстань: 1.0328)

**Шлях**: data/perceptron.md

# Introduction to Neural Networks: Perceptron One of the first attempts to implement something similar to a modern neural network was done by Frank Rosenblatt from Cornell Aeronautical Laboratory in 1957. It was a hardware implementation called "Mark-1", designed to recognize primitive geometric figures,



Результати збережено у файл: split_text_300_200.md


In [38]:
# Для довгих технічних текстів
splitted_df['chunks'] = splitted_df['text'].apply(lambda x: split_text(x, 800, 500))
compare_responses("Що таке згорткові нейронні мережі?", save_to_file=True, filename="split_text_800_500.md")


# Порівняння відповідей

## 📝 Запит: Що таке згорткові нейронні мережі?

## 🔍 Відповіді моделей

### Без використання RAG

Згорткові нейронні мережі (CNN, Convolutional Neural Networks) — це тип глибоких нейронних мереж, які спеціально розроблені для обробки даних з сітковою структурою, таких як зображення. Вони використовують операцію згортки для виявлення локальних ознак у даних, що дозволяє ефективно аналізувати просторові ієрархії. CNN складаються з кількох шарів, включаючи згорткові шари, шари активації, підвибірки (пулінг) та повнозв'язні шари. Вони широко використовуються в комп'ютерному зорі, розпізнаванні образів та інших завданнях, пов'язаних з обробкою зображень.

### З використанням RAG

Згорткові нейронні мережі (CNN) — це тип нейронних мереж, які спеціально розроблені для обробки даних з решітковою структурою, таких як зображення. Вони використовують згорткові шари для автоматичного виділення ознак з вхідних даних, що дозволяє ефективно виконувати завдання, пов'язані з класифікацією, розпізнаванням об'єктів та іншими задачами комп'ютерного зору. Згорткові мережі здатні виявляти складні патерни та структури в даних, що робить їх потужним інструментом у сфері штучного інтелекту.

## 📚 Знайдені фрагменти тексту


### Фрагмент 1 (відстань: 0.9587)

**Шлях**: data/own_framework.md

# Introduction to Neural Networks. Multi-Layered Perceptron In the previous section, you learned about the simplest neural network model - one-layered perceptron, a linear two-class classification model. In this section we will extend this model into a more flexible framework, allowing us to: * perform


### Фрагмент 2 (відстань: 0.9745)

**Шлях**: data/perceptron.md

array, so the neural network had 400 inputs and one binary output. A simple network contained one neuron, also called a **threshold logic unit**. Neural network weights acted like potentiometers that required manual adjustment during the training phase. > ✅ A potentiometer is a device that allows the


### Фрагмент 3 (відстань: 0.9998)

**Шлях**: data/own_framework.md

**multi-class classification** in addition to two-class * solve **regression problems** in addition to classification * separate classes that are not linearly separable We will also develop our own modular framework in Python that will allow us to construct different neural network architectures. ## Formalization


### Фрагмент 4 (відстань: 1.0301)

**Шлях**: data/own_framework.md

data. Because subset is taken randomly each time, such method is called **stochastic gradient descent** (SGD). ## Multi-Layered Perceptrons and Backpropagation One-layer network, as we have seen above, is capable of classifying linearly separable classes. To build a richer model, we can combine several


### Фрагмент 5 (відстань: 1.0328)

**Шлях**: data/perceptron.md

# Introduction to Neural Networks: Perceptron One of the first attempts to implement something similar to a modern neural network was done by Frank Rosenblatt from Cornell Aeronautical Laboratory in 1957. It was a hardware implementation called "Mark-1", designed to recognize primitive geometric figures,



Результати збережено у файл: split_text_800_500.md


### 4. Зміна кількості фрагментів - змініть параметр k у функції пошуку та дослідіть, як кількість знайдених фрагментів впливає на відповіді.

In [39]:
search_test("Переваги RAG", 2)
print("\n________________________________________________________________________________________________________________________________________________________________________")

search_test("Переваги RAG", 4)
print("\n________________________________________________________________________________________________________________________________________________________________________")

search_test("Переваги RAG", 1)

Фрагмент 1:
should give us the opportunity to compute gradients of *any expression* that we can define. Another important thing is to be able to perform computations on GPU, or any other specialized compute units, such as TPU. Deep neural network training requires *a lot* of computations, and to be able to parallelize
Шлях: data/frameworks.md
Відстань: 1.2010166645050049
--------------------------------------------------
Фрагмент 2:
**computational graphs**. This graph defines how to compute the output (usually the loss function) with given input parameters, and can be pushed for computation on GPU, if it is available. There are functions to differentiate this computational graph and compute gradients, which can then be used for
Шлях: data/frameworks.md
Відстань: 1.2197811603546143
--------------------------------------------------

___________________________________________________________________________________________________________________________________________________________

___

## Індивідуальне завдання варіант №3. Тема "Глибинне навчання"

### Частина 1: Базова реалізація RAG-системи на власних даних
Модифікуйте базовий ноутбук, щоб реалізувати RAG-систему для іншої тематики (згідно з вашим варіантом). Для цього:

1. Підготуйте 3-5 текстових документів з тематики вашого варіанту (в форматі .txt або .md);
2. Завантажте ці документи та створіть DataFrame з текстами;
3. Використайте функції з базового ноутбука для:
- Розбиття текстів на фрагменти;
- Створення ембедінгів;
- Налаштування векторного пошуку;
- Реалізації RAG чат-бота;
4. Протестуйте систему на 3-5 запитаннях, специфічних для вашої тематики;
5. Порівняйте відповіді RAG-системи з базовою моделлю.

In [40]:
my_data_paths = [
    "my_data/cnn.md", 
    "my_data/deap_learning.md", 
    "my_data/neural_network_base.md"
]

print(my_data_paths)

# Ініціалізація порожнього DataFrame
df = pd.DataFrame(columns=['path', 'text'])
# Сучасний спосіб додавання рядків до DataFrame
for path in my_data_paths:
    try:
        with open(path, 'r', encoding='utf-8') as file:
            file_content = file.read()
        # Використовуємо concat замість застарілого append
        new_row = pd.DataFrame({'path': [path], 'text': [file_content]})
        df = pd.concat([df, new_row], ignore_index=True)
    except FileNotFoundError:
        print(f"Файл не знайдено: {path}")
df.head()

['my_data/cnn.md', 'my_data/deap_learning.md', 'my_data/neural_network_base.md']


,path,text
0,my_data/cnn.md,# Згорткові нейронні мережі (CNN)\n\n## Вступ\...
1,my_data/deap_learning.md,# Що таке глибинне навчання?\n\n## Визначення\...
2,my_data/neural_network_base.md,НАЗВА: Нейронні мережі - основи архітектури\n\...


In [41]:
splitted_df = df.copy()
splitted_df['chunks'] = splitted_df['text'].apply(lambda x: split_text(x, 400, 300))

splitted_df

,path,text,chunks
0,my_data/cnn.md,# Згорткові нейронні мережі (CNN)\n\n## Вступ\...,[# Згорткові нейронні мережі (CNN) ## Вступ CN...
1,my_data/deap_learning.md,# Що таке глибинне навчання?\n\n## Визначення\...,[# Що таке глибинне навчання? ## Визначення Гл...
2,my_data/neural_network_base.md,НАЗВА: Нейронні мережі - основи архітектури\n\...,[НАЗВА: Нейронні мережі - основи архітектури О...


In [42]:
flattened_df = splitted_df.explode('chunks')

flattened_df.head()


,path,text,chunks
0,my_data/cnn.md,# Згорткові нейронні мережі (CNN)\n\n## Вступ\...,# Згорткові нейронні мережі (CNN) ## Вступ CNN...
0,my_data/cnn.md,# Згорткові нейронні мережі (CNN)\n\n## Вступ\...,"даних - Виявляє локальні особливості (краї, те..."
0,my_data/cnn.md,# Згорткові нейронні мережі (CNN)\n\n## Вступ\...,(Fully Connected Layer) - Аналогічний звичайни...
0,my_data/cnn.md,# Згорткові нейронні мережі (CNN)\n\n## Вступ\...,конкурсі - Використання ReLU та GPU ### VGGNet...
0,my_data/cnn.md,# Згорткові нейронні мережі (CNN)\n\n## Вступ\...,Faster R-CNN) - Сегментація зображень (U-Net) ...


In [43]:
flattened_df = get_or_create_embeddings(
    splitted_df.explode('chunks'), 
    'chunks', 
    create_embeddings,
    filename="my_data_embeddings.pkl"
)

print(flattened_df)

DataFrame успішно завантажено з embeddings\my_data_embeddings.pkl
                                                path  \
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
1  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
1  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
1  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
1  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
2  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
2  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
2  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
2  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   

                                                text  \
0  # Згорткові нейронні мережі (CNN)\n\n## Вступ\...   
0  # Згорткові нейронні мережі (CNN)\

In [44]:
# Отримуємо ембедінги як масив numpy
embeddings_list = flattened_df['embeddings'].to_list()
embeddings_array = np.array(embeddings_list).astype('float32')

# Визначаємо розмірність векторів
vector_dimension = len(embeddings_list[0])

# Створюємо індекс FAISS
index = faiss.IndexFlatL2(vector_dimension)  # L2 - це евклідова відстань

# Додаємо наші вектори до індексу
index.add(embeddings_array)

# Перевіряємо кількість векторів в індексі
print(f"Загальна кількість векторів в індексі: {index.ntotal}, розмірність векторів: {vector_dimension}")

Загальна кількість векторів в індексі: 13, розмірність векторів: 1024


In [45]:
search_test("Що таке нейромережі", 5)

Фрагмент 1:
Faster R-CNN) - Сегментація зображень (U-Net) - Генерація зображень (StyleGAN) - Обробка медичних зображень (діагностика)
Шлях: C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/1 семестр/Основи генеративного искуственого интелекта/generative-ai-for-beginners-nechko/Nechko/my_data/cnn.md
Відстань: 0.8103533387184143
--------------------------------------------------
Фрагмент 2:
НАЗВА: Нейронні мережі - основи архітектури ОСНОВНІ КОМПОНЕНТИ НЕЙРОННИХ МЕРЕЖ: 1. НЕЙРОНИ (ВУЗЛИ) - Базові обчислювальні одиниці - Приймають вхідні дані, обробляють їх і передають далі - Мають вагові коефіцієнти, які налаштовуються під час навчання 2. ШАРИ МЕРЕЖІ - Вхідний шар: отримує необроблені дані
Шлях: C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/1 семестр/Основи генеративного искуственого интелекта/generative-ai-for-beginners-nechko/Nechko/my_data/neural_network_base.md
Відстань: 0.8208187818527222
--------------------------------------------------
Фрагмент 3:
(NLP) - Генеративні моделі (GAN, ди

In [46]:
compare_responses("Нейромережі", save_to_file=True, filename="my_data_neural_networks.md")


# Порівняння відповідей

## 📝 Запит: Нейромережі

## 🔍 Відповіді моделей

### Без використання RAG

Нейромережі — це обчислювальні моделі, натхненні структурою та функціонуванням людського мозку. Вони складаються з взаємопов'язаних "нейронів", які обробляють інформацію. Нейромережі використовуються для розв'язання різноманітних задач, таких як розпізнавання образів, обробка природної мови, прогнозування та багато інших. Основні типи нейромереж включають:

1. **Прості нейронні мережі** (Feedforward Neural Networks) — дані проходять в одному напрямку.
2. **Згорткові нейронні мережі** (Convolutional Neural Networks, CNN) — використовуються переважно для обробки зображень.
3. **Рекурентні нейронні мережі** (Recurrent Neural Networks, RNN) — підходять для обробки послідовних даних, таких як текст або часо́ві ряди.

Нейромережі навчаються на основі великих обсягів даних, використовуючи алгоритми, такі як зворотне поширення помилки (backpropagation).

### З використанням RAG

Нейронні мережі є основою сучасних методів штучного інтелекту, зокрема в обробці зображень та природної мови. Основні компоненти нейронних мереж включають нейрони (вузли), які є базовими обчислювальними одиницями, що приймають, обробляють і передають вхідні дані, а також шари мережі, які складаються з вхідного шару, шару підвибірки (pooling layer) та повнозв'язного шару. Вхідний шар отримує необроблені дані, шар підвибірки зменшує просторові розміри, зберігаючи важливу інформацію, а повнозв'язний шар виконує фінальну обробку даних.

Серед архітектур нейронних мереж виділяються VGGNet, яка використовує прості 3×3 фільтри і має глибину до 19 шарів, та ResNet, що впроваджує залишкові зв'язки, дозволяючи тренувати дуже глибокі мережі (100+ шарів). Практичні застосування нейронних мереж охоплюють класифікацію зображень, детекцію об'єктів (наприклад,

## 📚 Знайдені фрагменти тексту


### Фрагмент 1 (відстань: 0.6948)

**Шлях**: C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/1 семестр/Основи генеративного искуственого интелекта/generative-ai-for-beginners-nechko/Nechko/my_data/cnn.md

Faster R-CNN) - Сегментація зображень (U-Net) - Генерація зображень (StyleGAN) - Обробка медичних зображень (діагностика)


### Фрагмент 2 (відстань: 0.7748)

**Шлях**: C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/1 семестр/Основи генеративного искуственого интелекта/generative-ai-for-beginners-nechko/Nechko/my_data/deap_learning.md

(NLP) - Генеративні моделі (GAN, дифузійні моделі) - Автономні транспортні засоби - Медична діагностика


### Фрагмент 3 (відстань: 0.8777)

**Шлях**: C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/1 семестр/Основи генеративного искуственого интелекта/generative-ai-for-beginners-nechko/Nechko/my_data/neural_network_base.md

НАЗВА: Нейронні мережі - основи архітектури ОСНОВНІ КОМПОНЕНТИ НЕЙРОННИХ МЕРЕЖ: 1. НЕЙРОНИ (ВУЗЛИ) - Базові обчислювальні одиниці - Приймають вхідні дані, обробляють їх і передають далі - Мають вагові коефіцієнти, які налаштовуються під час навчання 2. ШАРИ МЕРЕЖІ - Вхідний шар: отримує необроблені дані


### Фрагмент 4 (відстань: 0.8960)

**Шлях**: C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/1 семестр/Основи генеративного искуственого интелекта/generative-ai-for-beginners-nechko/Nechko/my_data/cnn.md

даних - Виявляє локальні особливості (краї, текстури, форми) - Параметри фільтрів навчаються під час тренування ### 2. Шар підвибірки (Pooling Layer) - Зменшує просторові розмірності - Зберігає важливу інформацію - Типи: Max Pooling, Average Pooling - Знижує ризик перенавчання ### 3. Повнозв'язний шар


### Фрагмент 5 (відстань: 0.9068)

**Шлях**: C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/1 семестр/Основи генеративного искуственого интелекта/generative-ai-for-beginners-nechko/Nechko/my_data/cnn.md

конкурсі - Використання ReLU та GPU ### VGGNet (2014) - Проста архітектура з 3×3 фільтрами - Глибина до 19 шарів ### ResNet (2015) - Залишкові зв'язки (skip connections) - Дозволила тренувати дуже глибокі мережі (100+ шарів) ## Практичні застосування - Класифікація зображень - Детекція об'єктів (YOLO,



Результати збережено у файл: my_data_neural_networks.md


___

### Частина 2: Розширене завдання
Розширте функціональність базового рішення, створивши інтерактивну систему для відповіді на запитання про навчальні курси. Така система матиме практичне застосування для навчальних закладів, освітніх платформ та онлайн-курсів.

Інтерактивний освітній RAG-асистент

Ключові аспекти реалізації:

1. Створення спеціалізованої бази знань
Зберіть дані про 5-10 курсів з вашої спеціальності або напрямку (згідно з варіантом), включивши для кожного:

- Назву та короткий опис курсу
- Перелік тем або модулів
- Вимоги до попередніх знань
- Практичні завдання
- Очікувані результати навчання
2. Реалізація спеціалізованих функцій
Створіть функції для обробки специфічних запитів:

In [48]:
my_data_paths = [
    "my_data/cnn.md", 
    "my_data/deap_learning.md", 
    "my_data/neural_network_base.md",
    "my_data/deap_learning_courses.md", 
    "my_data/aditional_courses.md"
    ]
print(my_data_paths)
# Ініціалізація порожнього DataFrame
df = pd.DataFrame(columns=['path', 'text'])
# Сучасний спосіб додавання рядків до DataFrame
for path in my_data_paths:
    try:
        with open(path, 'r', encoding='utf-8') as file:
            file_content = file.read()
        # Використовуємо concat замість застарілого append
        new_row = pd.DataFrame({'path': [path], 'text': [file_content]})
        df = pd.concat([df, new_row], ignore_index=True)
    except FileNotFoundError:
        print(f"Файл не знайдено: {path}")
df.head()
splitted_df = df.copy()
splitted_df['chunks'] = splitted_df['text'].apply(lambda x: split_text(x, 400, 300))

splitted_df
flattened_df = splitted_df.explode('chunks')

flattened_df.head()
flattened_df = get_or_create_embeddings(
    splitted_df.explode('chunks'), 
    'chunks', 
    create_embeddings,
    filename="my_data_new_embeddings.pkl"
)

print(flattened_df)

['my_data/cnn.md', 'my_data/deap_learning.md', 'my_data/neural_network_base.md', 'my_data/deap_learning_courses.md', 'my_data/aditional_courses.md']
Файл не знайдено: my_data/deap_learning_courses.md
Файл не знайдено: my_data/aditional_courses.md
DataFrame успішно завантажено з embeddings\my_data_new_embeddings.pkl
                                                path  \
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
0  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
1  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
1  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
1  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
1  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
2  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
2  C:/Users/diman/Desktop/Коледж/Бакалавр/1 курс/...   
2  C:/Users

In [49]:
def find_course_by_topic(user_query):
    """
    Знаходить курси, що відповідають запиту користувача про певну тему.
    
    Args:
        user_query: Запит користувача про бажану тему навчання
    
    Returns:
        Структурована відповідь з рекомендованими курсами
    """
    # Пошук релевантних фрагментів
    query_vector = create_embeddings(user_query)
    query_vector_array = np.array([query_vector]).astype('float32')
    distances, indices = index.search(query_vector_array, 5)
    
    # Збір інформації про курси з найбільш релевантних фрагментів
    course_info = []
    course_names = set()  # Для виключення дублікатів
    
    for idx in indices[0]:
        chunk = flattened_df['chunks'].iloc[idx]
        # Витягуємо назву курсу з фрагменту
        # Це можна зробити через регулярні вирази або простий пошук ключових фраз
        import re
        course_match = re.search(r'Курс[:\s]+([^\n.]+)', chunk)
        if course_match and course_match.group(1) not in course_names:
            course_name = course_match.group(1).strip()
            course_names.add(course_name)
            
            # Збираємо додаткову інформацію для цього курсу
            course_info.append({
                "name": course_name,
                "relevance": float(1.0 / (1.0 + distances[0][indices[0].tolist().index(idx)])),
                "context": chunk
            })
    
    # Підготовка контексту для моделі
    context = "\n\n".join([f"Курс: {info['name']}\nОпис: {info['context']}" for info in course_info])
    
    # Формування відповіді через LLM
    messages = [
        {"role": "system", "content": "Ви освітній асистент, який допомагає знаходити відповідні курси навчання."},
        {"role": "user", "content": f"На основі наступної інформації про курси:\n\n{context}\n\n"
                                    f"Порекомендуйте найбільш підходящі курси для запиту: '{user_query}'. "
                                    f"Дайте структуровану відповідь з назвами курсів та коротким описом, чому саме ці курси підходять."}
    ]
    
    response = client.complete(
        temperature=0.3,
        model=deployment,
        messages=messages,
        max_tokens=400,
    )
    
    return response.choices[0].message.content

def answer_course_question(course_name, user_question):
    """
    Відповідає на конкретні запитання про певний курс.
    
    Args:
        course_name: Назва курсу
        user_question: Запитання користувача
    
    Returns:
        Детальна відповідь на запитання
    """
    # Формуємо запит, що комбінує назву курсу та запитання
    combined_query = f"Курс: {course_name}. Запитання: {user_question}"
    
    # Пошук релевантних фрагментів
    query_vector = create_embeddings(combined_query)
    query_vector_array = np.array([query_vector]).astype('float32')
    distances, indices = index.search(query_vector_array, 3)  # Беремо менше фрагментів для конкретнішої відповіді
    
    # Збираємо контекст
    context_fragments = [flattened_df['chunks'].iloc[idx] for idx in indices[0]]
    context = "\n\n".join(context_fragments)
    
    # Формування відповіді
    messages = [
        {"role": "system", "content": "Ви освітній асистент, який відповідає на запитання про навчальні курси."},
        {"role": "user", "content": f"Інформація про курс '{course_name}':\n\n{context}\n\n"
                                    f"Дайте детальну відповідь на запитання: '{user_question}', "
                                    f"використовуючи тільки надану інформацію. Якщо інформації недостатньо, "
                                    f"чесно визнайте це."}
    ]
    
    response = client.complete(
        temperature=0.2,
        model=deployment,
        messages=messages,
        max_tokens=300,
    )
    
    return response.choices[0].message.content

3. Створення інтерактивного інтерфейсу
Реалізуйте просту систему меню для взаємодії:

In [50]:
def educational_assistant():
    """
    Інтерактивний освітній асистент з меню вибору функцій.
    """
    from IPython.display import display, Markdown, clear_output
    import ipywidgets as widgets
    
    # Створюємо віджети для взаємодії
    query_text = widgets.Textarea(
        value='',
        placeholder='Введіть ваш запит...',
        description='Запит:',
        disabled=False,
        layout=widgets.Layout(width='100%', height='80px')
    )
    
    course_text = widgets.Text(
        value='',
        placeholder='Назва курсу (для детальних запитань)',
        description='Курс:',
        disabled=False,
        layout=widgets.Layout(width='100%')
    )
    
    action_dropdown = widgets.Dropdown(
        options=[
            ('Знайти курси за темою', 'find'),
            ('Запитати про конкретний курс', 'ask')
        ],
        value='find',
        description='Дія:',
        disabled=False,
    )
    
    output = widgets.Output()
    
    def on_button_clicked(b):
        with output:
            clear_output()
            
            if action_dropdown.value == 'find':
                if not query_text.value.strip():
                    display(Markdown("❌ Будь ласка, введіть запит про бажану тему навчання."))
                    return
                
                display(Markdown(f"## Пошук курсів за темою: {query_text.value}"))
                response = find_course_by_topic(query_text.value)
                display(Markdown(response))
                
            elif action_dropdown.value == 'ask':
                if not course_text.value.strip() or not query_text.value.strip():
                    display(Markdown("❌ Будь ласка, введіть назву курсу та ваше запитання."))
                    return
                
                display(Markdown(f"## Запитання про курс: {course_text.value}"))
                display(Markdown(f"### Запитання: {query_text.value}"))
                response = answer_course_question(course_text.value, query_text.value)
                display(Markdown("### Відповідь:"))
                display(Markdown(response))
    
    button = widgets.Button(description="Отримати відповідь")
    button.on_click(on_button_clicked)
    
    # Компонуємо інтерфейс
    display(Markdown("# 🎓 Освітній RAG-асистент"))
    display(Markdown("Цей асистент допоможе знайти відповідні курси або відповісти на запитання про конкретний курс."))
    display(action_dropdown)
    display(query_text)
    display(course_text)
    display(button)
    display(output)

In [ ]:
educational_assistant()

# 🎓 Освітній RAG-асистент

Цей асистент допоможе знайти відповідні курси або відповісти на запитання про конкретний курс.

Dropdown(description='Дія:', options=(('Знайти курси за темою', 'find'), ('Запитати про конкретний курс', 'ask…

Textarea(value='', description='Запит:', layout=Layout(height='80px', width='100%'), placeholder='Введіть ваш …

Text(value='', description='Курс:', layout=Layout(width='100%'), placeholder='Назва курсу (для детальних запит…

Button(description='Отримати відповідь', style=ButtonStyle())

Output()

4. Тестування й оцінка
Протестуйте систему з:

- Запитами про загальні теми ("Які курси допоможуть вивчити Python?")
- Конкретними запитаннями про курси ("Які практичні завдання є в курсі Machine Learning?")
- Нестандартними запитами, щоб перевірити обмеження системи

5. Додаткові функції (за бажанням):

- Рейтингування курсів за релевантністю до запиту
- Генерація індивідуального плану навчання на основі зацікавлень користувача
- Підтримка фільтрації за складністю курсів

Така система має пряме практичне застосування у освітній сфері і може стати основою для розробки більш комплексних освітніх асистентів, систем рекомендації курсів і навіть інтерактивних навчальних платформ.